In [10]:
from pathlib import Path

base_path = Path.cwd()

matches = list(base_path.rglob("salary_task1_cleaned(2).csv"))

print("Files found:")
for file in matches:
    print(file)

from pathlib import Path

base_path = Path.cwd()

matches = list(base_path.rglob("*salary*cleaned*.csv"))

print("Matching CSV files:")
for file in matches:
    print(file)

Files found:
Matching CSV files:
C:\Users\DELL\OneDrive\New folder\Desktop\brainybeam\18-09-26 tasks\salary_task1_cleaned.csv


In [11]:
import pandas as pd

file_path = matches[0]

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (2000, 18)

Columns:
['age', 'gender', 'education', 'experience_years', 'role_seniority', 'company_size', 'location_tier', 'skills_count', 'certifications', 'worked_remote', 'last_promotion_years_ago', 'salary_bdt', 'recent_project_description_length', 'survey_date', 'recent_note', 'survey_year', 'survey_month', 'role_seniority_encoded']

Data Types:
age                                    int64
gender                                   str
education                                str
experience_years                     float64
role_seniority                           str
company_size                             str
location_tier                            str
skills_count                         float64
certifications                       float64
worked_remote                          int64
last_promotion_years_ago             float64
salary_bdt                             int64
recent_project_description_length    float64
survey_date                              str
re

In [12]:
# STEP 2: Identify Categorical Variables
categorical_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("All object/string columns:")
print(categorical_columns)
for col in categorical_columns:
    print(f"\nColumn: {col}")
    print(f"Unique values: {df[col].nunique(dropna=False)}")
    print(f"Missing values: {df[col].isna().sum()}")
    print("Values:")
    print(df[col].value_counts(dropna=False).head(15))

All object/string columns:
['gender', 'education', 'role_seniority', 'company_size', 'location_tier', 'survey_date', 'recent_note']

Column: gender
Unique values: 3
Missing values: 0
Values:
gender
Male      1437
Female     530
Other       33
Name: count, dtype: int64

Column: education
Unique values: 6
Missing values: 100
Values:
education
B.Sc         1070
M.Sc          445
B.Sc+Cert     229
NaN           100
M.Eng          99
PhD            57
Name: count, dtype: int64

Column: role_seniority
Unique values: 4
Missing values: 0
Values:
role_seniority
Mid       791
Junior    715
Senior    393
Lead      101
Name: count, dtype: int64

Column: company_size
Unique values: 3
Missing values: 0
Values:
company_size
SME           810
Startup       596
Enterprise    594
Name: count, dtype: int64

Column: location_tier
Unique values: 4
Missing values: 0
Values:
location_tier
Tier-1    813
Tier-2    620
Remote    321
Tier-3    246
Name: count, dtype: int64

Column: survey_date
Unique values: 691

In [13]:
# STEP 4: Rare Category Analysis

encoding_candidates = [
    "gender",
    "education",
    "role_seniority",
    "company_size",
    "location_tier"
]
for col in encoding_candidates:
    counts = df[col].value_counts(dropna=False)
    percentages = df[col].value_counts(
        normalize=True,
        dropna=False
    ) * 100
    
    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })
    
    print(summary)

        Count  Percentage
gender                   
Male     1437       71.85
Female    530       26.50
Other      33        1.65
           Count  Percentage
education                   
B.Sc        1070       53.50
M.Sc         445       22.25
B.Sc+Cert    229       11.45
NaN          100        5.00
M.Eng         99        4.95
PhD           57        2.85
                Count  Percentage
role_seniority                   
Mid               791       39.55
Junior            715       35.75
Senior            393       19.65
Lead              101        5.05
              Count  Percentage
company_size                   
SME             810        40.5
Startup         596        29.8
Enterprise      594        29.7
               Count  Percentage
location_tier                   
Tier-1           813       40.65
Tier-2           620       31.00
Remote           321       16.05
Tier-3           246       12.30


In [15]:
# STEP 5: Validate Existing Role Seniority Encoding

print(
    df[
        ["role_seniority", "role_seniority_encoded"]
    ]
    .drop_duplicates()
    .sort_values("role_seniority_encoded")
)

  role_seniority  role_seniority_encoded
1         Junior                       0
3            Mid                       1
0         Senior                       2
2           Lead                       3


In [17]:
# STEP 6: Category Distribution Check
one_hot_columns = [
    "gender",
    "education",
    "company_size",
    "location_tier"
]

for col in one_hot_columns:
    print(
        df[col]
        .value_counts(dropna=False)
        .to_frame("Count")
    )

        Count
gender       
Male     1437
Female    530
Other      33
           Count
education       
B.Sc        1070
M.Sc         445
B.Sc+Cert    229
NaN          100
M.Eng         99
PhD           57
              Count
company_size       
SME             810
Startup         596
Enterprise      594
               Count
location_tier       
Tier-1           813
Tier-2           620
Remote           321
Tier-3           246


In [18]:
# STEP 6: One-Hot Encoding

one_hot_columns = [
    "gender",
    "education",
    "company_size",
    "location_tier"
]

# Create a copy so original dataset remains unchanged
df_encoded = df.copy()

# Convert missing education into an explicit category
df_encoded["education"] = df_encoded["education"].fillna("Missing")

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(
    df_encoded,
    columns=one_hot_columns,
    dtype=int
)

print("Original Dataset Shape:", df.shape)
print("Encoded Dataset Shape:", df_encoded.shape)

print("\nNew Columns:")
for col in df_encoded.columns:
    print(col)

Original Dataset Shape: (2000, 18)
Encoded Dataset Shape: (2000, 30)

New Columns:
age
experience_years
role_seniority
skills_count
certifications
worked_remote
last_promotion_years_ago
salary_bdt
recent_project_description_length
survey_date
recent_note
survey_year
survey_month
role_seniority_encoded
gender_Female
gender_Male
gender_Other
education_B.Sc
education_B.Sc+Cert
education_M.Eng
education_M.Sc
education_Missing
education_PhD
company_size_Enterprise
company_size_SME
company_size_Startup
location_tier_Remote
location_tier_Tier-1
location_tier_Tier-2
location_tier_Tier-3


In [19]:
# STEP 7: Verify Encoded Data
print("Encoded categorical columns:")
encoded_columns = [
    col for col in df_encoded.columns
    if any(col.startswith(prefix + "_") for prefix in one_hot_columns)
]
print(encoded_columns)
print("\nFirst 5 rows:")
display(df_encoded[encoded_columns].head())

Encoded categorical columns:
['gender_Female', 'gender_Male', 'gender_Other', 'education_B.Sc', 'education_B.Sc+Cert', 'education_M.Eng', 'education_M.Sc', 'education_Missing', 'education_PhD', 'company_size_Enterprise', 'company_size_SME', 'company_size_Startup', 'location_tier_Remote', 'location_tier_Tier-1', 'location_tier_Tier-2', 'location_tier_Tier-3']

First 5 rows:


,gender_Female,gender_Male,gender_Other,education_B.Sc,education_B.Sc+Cert,education_M.Eng,education_M.Sc,education_Missing,education_PhD,company_size_Enterprise,company_size_SME,company_size_Startup,location_tier_Remote,location_tier_Tier-1,location_tier_Tier-2,location_tier_Tier-3
0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0
1,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0
2,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0
3,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0
4,1,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0


In [20]:
# STEP 8: Remaining Object Columns
remaining_object_columns = df_encoded.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Remaining object/string columns:")
print(remaining_object_columns)

Remaining object/string columns:
['role_seniority', 'survey_date', 'recent_note']


In [21]:
# STEP 9: Missing Values After Encoding
missing_after_encoding = df_encoded.isnull().sum()
print(
    missing_after_encoding[
        missing_after_encoding > 0
    ]
)

experience_years                     160
skills_count                         200
certifications                        80
last_promotion_years_ago             120
recent_project_description_length    240
recent_note                          300
dtype: int64


In [22]:
# STEP 10: High-Cardinality Analysis

print("Cardinality of categorical variables:")
for col in ["gender", "education", "role_seniority", 
            "company_size", "location_tier"]:
    
    unique_count = df[col].nunique(dropna=False)
    
    print(f"{col:20} : {unique_count} unique values")

Cardinality of categorical variables:
gender               : 3 unique values
education            : 6 unique values
role_seniority       : 4 unique values
company_size         : 3 unique values
location_tier        : 4 unique values


In [23]:
# STEP 12: Final Encoding Validation
print("Original shape :", df.shape)
print("Encoded shape  :", df_encoded.shape)
print("\nEncoded categorical columns:")
for col in df_encoded.columns:
    if (
        col.startswith("gender_")
        or col.startswith("education_")
        or col.startswith("company_size_")
        or col.startswith("location_tier_")
    ):
        print(col)

Original shape : (2000, 18)
Encoded shape  : (2000, 30)

Encoded categorical columns:
gender_Female
gender_Male
gender_Other
education_B.Sc
education_B.Sc+Cert
education_M.Eng
education_M.Sc
education_Missing
education_PhD
company_size_Enterprise
company_size_SME
company_size_Startup
location_tier_Remote
location_tier_Tier-1
location_tier_Tier-2
location_tier_Tier-3


In [24]:
# Check data types of encoded columns
print("\nData types of newly encoded columns:")

encoded_cols = [
    col for col in df_encoded.columns
    if (
        col.startswith("gender_")
        or col.startswith("education_")
        or col.startswith("company_size_")
        or col.startswith("location_tier_")
    )
]

print(df_encoded[encoded_cols].dtypes)


Data types of newly encoded columns:
gender_Female              int64
gender_Male                int64
gender_Other               int64
education_B.Sc             int64
education_B.Sc+Cert        int64
education_M.Eng            int64
education_M.Sc             int64
education_Missing          int64
education_PhD              int64
company_size_Enterprise    int64
company_size_SME           int64
company_size_Startup       int64
location_tier_Remote       int64
location_tier_Tier-1       int64
location_tier_Tier-2       int64
location_tier_Tier-3       int64
dtype: object


In [25]:
# STEP 13: FINAL ENCODING VALIDATION

# 1. Dataset shape
print("\n1. Dataset Shape")
print("Original :", df.shape)
print("Encoded  :", df_encoded.shape)

# 2. Duplicate rows
print("\n2. Duplicate Rows")
print("Duplicates:", df_encoded.duplicated().sum())

# 3. Remaining categorical/object columns
print("\n3. Remaining Object/String Columns")
print(df_encoded.select_dtypes(include=["object", "string"]).columns.tolist())

# 4. Newly encoded columns
encoded_cols = [
    col for col in df_encoded.columns
    if (
        col.startswith("gender_")
        or col.startswith("education_")
        or col.startswith("company_size_")
        or col.startswith("location_tier_")
    )
]

print("\n4. Number of New Encoded Columns")
print(len(encoded_cols))

# 5. Encoded columns data types
print("\n5. Encoded Columns Data Types")
print(df_encoded[encoded_cols].dtypes.value_counts())

# 6. Missing values
print("\n6. Columns Having Missing Values")
missing = df_encoded.isnull().sum()
print(missing[missing > 0])

# 7. Final preview
print("\n7. Final Dataset Preview")
display(df_encoded.head())


1. Dataset Shape
Original : (2000, 18)
Encoded  : (2000, 30)

2. Duplicate Rows
Duplicates: 0

3. Remaining Object/String Columns
['role_seniority', 'survey_date', 'recent_note']

4. Number of New Encoded Columns
16

5. Encoded Columns Data Types
int64    16
Name: count, dtype: int64

6. Columns Having Missing Values
experience_years                     160
skills_count                         200
certifications                        80
last_promotion_years_ago             120
recent_project_description_length    240
recent_note                          300
dtype: int64

7. Final Dataset Preview


,age,experience_years,role_seniority,skills_count,certifications,worked_remote,last_promotion_years_ago,salary_bdt,recent_project_description_length,survey_date,...,education_M.Sc,education_Missing,education_PhD,company_size_Enterprise,company_size_SME,company_size_Startup,location_tier_Remote,location_tier_Tier-1,location_tier_Tier-2,location_tier_Tier-3
0,33,6.0,Senior,1.0,2.0,1,0.0,145185,57.0,2024-11-15,...,1,0,0,1,0,0,0,0,1,0
1,29,9.0,Junior,5.0,0.0,1,0.0,121262,50.0,2024-03-08,...,1,0,0,1,0,0,1,0,0,0
2,34,NaN,Lead,5.0,1.0,1,5.0,184875,50.0,2024-09-01,...,0,0,0,1,0,0,1,0,0,0
3,39,16.0,Mid,5.0,0.0,1,3.0,180105,59.0,2024-01-26,...,0,0,0,1,0,0,0,1,0,0
4,29,8.0,Junior,5.0,2.0,0,7.0,114750,NaN,2023-12-08,...,0,0,0,0,1,0,0,1,0,0


In [26]:
# STEP 14: SAVE TASK 1 ENCODED DATASET

output_file = "salary_task1_encoded.csv"

df_encoded.to_csv(
    output_file,
    index=False
)

print(f"Task 1 encoded dataset saved successfully as: {output_file}")

Task 1 encoded dataset saved successfully as: salary_task1_encoded.csv


In [27]:
# Verify saved file
import os

print("File exists:", os.path.exists(output_file))
print("File size:", os.path.getsize(output_file), "bytes")

File exists: True
File size: 240793 bytes
